In [ ]:
import os
import clickhouse_connect
import json
import glob
import dotenv
from openai import OpenAI

In [ ]:
# load environment variables
dotenv.load_dotenv()
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST")
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE")
INFERENCE_BASE_URL = os.getenv("INFERENCE_ENDPOINT")
INFERENCE_API_KEY = os.getenv("INFERENCE_KEY")
INFERENCE_MODEL = os.getenv("INFERENCE_MODEL")

In [ ]:
_ch_host, _ch_port = CLICKHOUSE_HOST.split(":")
client_ch = clickhouse_connect.get_client(
    host=_ch_host,
    port=int(_ch_port),
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

def query_clickhouse(sql):
    result = client_ch.query(sql)
    columns = result.column_names
    return {"data": [dict(zip(columns, row)) for row in result.result_rows]}

In [ ]:
# explore response_payload column structure
schema = query_clickhouse("DESCRIBE TABLE query_logs")
for col in schema['data']:
    print(f"{col['name']:40s} {col['type']}")

In [ ]:
# peek at payload
sample = query_clickhouse("""
SELECT id, response_payload
FROM query_logs
WHERE length(toString(response_payload)) > 100
LIMIT 1
""")
row = sample['data'][0]
print("id:", row['id'])
print("response_payload type:", type(row['response_payload']))
print("response_payload:")
print(json.dumps(row['response_payload'], indent=2)[:2000])

In [ ]:
# load all 01-*.json range files
range_files = sorted(glob.glob("01-*.json"))
print(f"Found range files: {range_files}")

ranges = []
for f in range_files:
    with open(f) as fp:
        data = json.load(fp)
    ranges.append({"file": f, "startId": data["startId"], "endId": data["endId"]})
    print(f"  {f}: {data['startId']} -> {data['endId']}")

In [ ]:
# query clickhouse for all logs within the 01-*.json ranges
range_clauses = " OR ".join(
    f"(id >= {r['startId']} AND id <= {r['endId']})"
    for r in ranges
)

result = query_clickhouse(f"""
SELECT id, model, toString(request_payload) AS request_payload
FROM query_logs
WHERE {range_clauses}
ORDER BY id ASC
""")
logs = result['data']
# parse request_payload strings back to dicts
for r in logs:
    if isinstance(r['request_payload'], str):
        try:
            r['request_payload'] = json.loads(r['request_payload'])
        except json.JSONDecodeError:
            r['request_payload'] = {}

print(f"Fetched {len(logs)} logs")

In [ ]:
def extract_last_user_text(request_payload):
    """Extract last user message text for preview"""
    messages = request_payload.get("messages", [])
    for msg in reversed(messages):
        if msg.get("role") == "user":
            content = msg.get("content", "")
            if isinstance(content, str):
                return content
            if isinstance(content, list):
                for block in content:
                    if isinstance(block, dict) and block.get("type") == "text":
                        return block.get("text", "")
    return ""

# preview
for r in logs[:3]:
    msgs = r['request_payload'].get("messages", [])
    preview = extract_last_user_text(r['request_payload'])
    print(f"=== id={r['id']} ===")
    print(f"total messages: {len(msgs)}, will send last 5")
    print(f"last user msg: {preview[:300]}")

In [ ]:
# set up client
client = OpenAI(
    base_url=INFERENCE_BASE_URL,
    api_key=INFERENCE_API_KEY,
)

CATEGORIES = ["explore", "implement", "test", "debug", "prompt"]

SYSTEM_PROMPT = """You are a classifier for AI coding assistant requests.
Given the request payload JSON, classify the AI's current action into exactly one of these categories:

- explore: Reading/exploring code, understanding architecture, listing files, examining structure, gathering information
- implement: Writing new code, implementing features, creating files, refactoring existing code
- test: Writing tests, running tests, verifying functionality, checking test results
- debug: Fixing bugs, inv0estigating errors, tracing issues, diagnosing problems
- prompt: Telling the user 
" clarifying question, requesting more information, waiting for user input

Respond with ONLY the single category word, nothing else."""

def classify_response(request_payload):
    payload = {
        k: (v[-5:] if k == "messages" else v)
        for k, v in request_payload.items()
        if k != "tools"
    }
    payload_json = json.dumps(payload, ensure_ascii=False)

    resp = client.chat.completions.create(
        model=INFERENCE_MODEL,
        max_tokens=512,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Classify this AI coding assistant request:\n\n```{payload_json}``"}
        ]
    )
    category = resp.choices[0].message.content.strip().lower()
    return category

print("Classifier ready.")

In [ ]:
# load existing categories.json
CATEGORIES_PATH = "categories.json"
try:
    with open(CATEGORIES_PATH) as f:
        categories = json.load(f)
    print(f"Loaded {len(categories)} existing entries from {CATEGORIES_PATH}")
except (FileNotFoundError, json.JSONDecodeError):
    categories = {}
    print("No existing categories.json, starting fresh")

pending = [r for r in logs if str(r['id']) not in categories]
print(f"Pending: {len(pending)} / {len(logs)} logs need classification")

In [ ]:
for i, r in enumerate(pending):
    log_id = str(r['id'])
    category = classify_response(r['request_payload'])
    categories[log_id] = category
    print(f"[{i+1:2d}/{len(pending)}] {log_id}  -> {category}")

print(f"\nDone. Total classified: {len(categories)} logs.")

In [ ]:
# summary
from collections import Counter
counts = Counter(categories.values())
print("Category distribution:")
for cat, n in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "#" * n
    print(f"  {cat:12s}: {n:3d}  {bar}")

In [ ]:
# save to categories.json
output_path = "categories.json"
with open(output_path, "w") as f:
    json.dump(categories, f, indent=2)

print(f"Saved {len(categories)} entries to {output_path}")
print(json.dumps(categories, indent=2))